# Preprocessing

In [26]:
import pandas as pd

In [27]:
# Load data
df = pd.read_csv("Deidentified-Filtered Numeric Data_Yes and No 3 Online Classes.csv", skiprows=[1])

In [28]:
# Averaging and rounding the items 
df["AU"] = df[["AU 1", "AU 2", "AU 3", "AU 4"]].mean(axis=1).round().astype(int) # Actual Use
df["BI"] = df[["BI 1", "BI 2", "BI 3", "BI 4"]].mean(axis=1).round().astype(int) # Behavioral Intention
df["PU"] = df[["PU 1", "PU 2", "Repeat of PU 2", "PU 3", "PU 4"]].mean(axis=1).round().astype(int) # Perceived Usefulness
df["PEOU"] = df[["PEOU 1", "PEOU 2", "PEOU 3", "PEOU 4"]].mean(axis=1).round().astype(int) # Perceived Ease of Use
df["COV"] = df[["PSOC/VOL 1", "PSOC/VOL 2", "PSOC/VOL 3", "PSOC/VOL 4"]].mean(axis=1).round().astype(int) #COVID-19 Impact
df["COM"] = df[["CP/PI 1", "CP/PI 2", "CP/PI 3", "CP/PI 4"]].mean(axis=1).round().astype(int) #Compatibility
df["IQ"] = df[["IQ 1", "IQ 2", "IQ 3", "IQ 4", "IQ 5"]].mean(axis=1).round().astype(int) # Information Quality
df["SQ"] = df[["SQ/SYQ/AC 1", "SQ/SYQ/AC 2", "SQ/SYQ/AC 3", "SQ/SYQ/AC 4"]].mean(axis=1).round().astype(int) # System Quality
df["SCSE"] = df[["SCSE 1", "SCSE 2", "SCSE 3", "SCSE 4"]].mean(axis=1).round().astype(int) # Student Computer Self-efficacy

In [29]:
# Binning the averages into three states
def bin_likert(v):
    if v <= 2:
        return 0 # Low
    elif v == 3 or v == 4:
        return 1 # Medium
    else:
        return 2 # High
        
# Apply the binning to selected columns
cols = ["AU","BI","PU","PEOU","COV","COM","IQ","SQ","SCSE"]

for c in cols:
    df[c] = df[c].apply(bin_likert)

# CPTs

In [30]:
# function for probability (k=3 categories)
def calculate_probability(counts, k=3):
    for i in range(k):
        if i not in counts:
            counts.loc[i] = 0
    counts = counts.sort_index()
    return counts / counts.sum()

In [42]:
labels = {0: "Low", 1:"Medium", 2:"High"}

# Prior Probabilities
# COV
cov_counts = df["COV"].value_counts().sort_index()
P_cov = calculate_probability(cov_counts)
cov_list = [[round(p, 4)] for p in P_cov.values]

# COM
com_counts = df["COM"].value_counts().sort_index()
P_com = calculate_probability(com_counts)
com_list = [[round(p, 4)] for p in P_com.values]

# IQ
iq_counts = df["IQ"].value_counts().sort_index()
P_iq = calculate_probability(iq_counts)
iq_list = [[round(p, 4)] for p in P_iq.values]

# SQ
sq_counts = df["SQ"].value_counts().sort_index()
P_sq = calculate_probability(sq_counts)
sq_list = [[round(p, 4)] for p in P_sq.values]

# SCSE
scse_counts = df["SCSE"].value_counts().sort_index()
P_scse = calculate_probability(scse_counts)
scse_list = [[round(p, 4)] for p in P_scse.values]

In [7]:
# Conditional Probabilities

# P(PEOU | COV, COM, IQ, SQ, SCSE)
results = []

for cov in range(3):
    for com in range(3):
        for iq in range(3):
            for sq in range(3):
                for scse in range(3):

                    cond = (df["COV"] == cov) & (df["COM"] == com) & (df["IQ"] == iq) & \
                           (df["SQ"] == sq) & (df["SCSE"] == scse)

                    temp = df[cond]

                    if len(temp) == 0:
                        for peou in range(3):
                            results.append({
                                "COV": cov,
                                "COM": com,
                                "IQ": iq,
                                "SQ": sq,
                                "SCSE": scse,
                                "PEOU": peou,
                                "Probability": 0.33
                            })
                    else:
                        counts = temp["PEOU"].value_counts().sort_index()
                        probs = calculate_probability(counts)

                        for val in probs.index:
                            results.append({
                                "COV": cov,
                                "COM": com,
                                "IQ": iq,
                                "SQ": sq,
                                "SCSE": scse,
                                "PEOU": val,
                                "Probability": probs[val]
                            })

results_df = pd.DataFrame(results)

formatted_df = results_df.pivot_table(
    index=["COV", "COM", "IQ", "SQ", "SCSE"],
    columns="PEOU",
    values="Probability",
    fill_value=0
).reset_index()

formatted_df.columns.name = None

formatted_df = formatted_df.rename(columns={
    0: "P(PEOU=Low)",
    1: "P(PEOU=Medium)",
    2: "P(PEOU=High)"
})

formatted_df.to_csv("PEOU.csv", index=False)


# P(PU | COV, COM, IQ, SQ, SCSE, PEOU)
results = []

for cov in range(3):
    for com in range(3):
        for iq in range(3):
            for sq in range(3):
                for scse in range(3):
                    for peou in range(3):

                        cond = (df["COV"] == cov) & (df["COM"] == com) & (df["IQ"] == iq) & \
                               (df["SQ"] == sq) & (df["SCSE"] == scse) & (df["PEOU"] == peou)

                        temp = df[cond]

                        if len(temp) == 0:
                            for pu_val in range(3):
                                results.append({
                                    "COV": cov,
                                    "COM": com,
                                    "IQ": iq,
                                    "SQ": sq,
                                    "SCSE": scse,
                                    "PEOU": peou,
                                    "PU": pu_val,
                                    "Probability": 0.33
                                })
                        else:
                            counts = temp["PU"].value_counts().sort_index()
                            probs = calculate_probability(counts)

                            for pu_val in probs.index:
                                results.append({
                                    "COV": cov,
                                    "COM": com,
                                    "IQ": iq,
                                    "SQ": sq,
                                    "SCSE": scse,
                                    "PEOU": peou,
                                    "PU": pu_val,
                                    "Probability": probs[pu_val]
                                })

results_df = pd.DataFrame(results)

formatted_df = results_df.pivot_table(
    index=["COV", "COM", "IQ", "SQ", "SCSE", "PEOU"],
    columns="PU",
    values="Probability",
    fill_value=0
).reset_index()

formatted_df.columns.name = None

formatted_df = formatted_df.rename(columns={
    0: "P(PU=Low)",
    1: "P(PU=Medium)",
    2: "P(PU=High)"
})

formatted_df.to_csv("PU.csv", index=False)

In [64]:
# P(BI | PU, PEOU)
results = []

for pu in range(3):
    for peou in range(3):

        condition = (df["PU"] == pu) & (df["PEOU"] == peou)
        subset = df[condition]

        if subset.empty:
            for bi in range(3):
                results.append({
                    "PU": pu,
                    "PEOU": peou,
                    "BI": bi,
                    "Probability": 0.33
                })
        else:
            counts = subset["BI"].value_counts().sort_index()
            probs = calculate_probability(counts)

            for bi_val in probs.index:
                results.append({
                    "PU": pu,
                    "PEOU": peou,
                    "BI": bi_val,
                    "Probability": probs[bi_val]
                })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["BI", "PU", "PEOU"])

bi_lists = []

for bi in range(3):
    bi_lists.append(results_df[results_df["BI"] == bi]["Probability"].round(4).tolist())

# P(AU | BI)
au_lists = []

for au_val in range(3):
    row = []
    for bi_val in range(3):
        data_bi = df[df["BI"] == bi_val]
        total = len(data_bi)
        # Count how many have AU = au_val
        count = len(data_bi[data_bi["AU"] == au_val])
        if total > 0:
            p = count / total
        else:
            p = 0
        row.append(round(p, 4))
    au_lists.append(row)

# Print the final conditional probability table
print(au_lists)

[[0.2917, 0.0355, 0.0], [0.6042, 0.8879, 0.3077], [0.1042, 0.0766, 0.6923]]


In [19]:
# Convert the P(PEOU | COV, COM, IQ, SQ, SCSE) to list
df = pd.read_csv("PEOU.csv")
peou_low_values = df['P(PEOU=Low)'].tolist()
peou_med_values = df['P(PEOU=Medium)'].tolist()
peou_high_values = df['P(PEOU=High)'].tolist()

# Convert the # P(PU | COV, COM, IQ, SQ, SCSE, PEOU) to list
df = pd.read_csv("PU.csv")
PU_low_values = df['P(PU=Low)'].tolist()
PU_med_values = df['P(PU=Medium)'].tolist()
PU_high_values = df['P(PU=High)'].tolist()

# Bayesian Network & Inference

In [20]:
# Defining the Bayesian Network

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
import itertools

In [65]:
# Defining the model structure
model = DiscreteBayesianNetwork([
    ('COVID_Impact', 'Perceived_Ease_of_Use'),
    ('Compatibility', 'Perceived_Ease_of_Use'),
    ('Information_Quality', 'Perceived_Ease_of_Use'),
    ('System_Quality', 'Perceived_Ease_of_Use'),
    ('Student_Comp_SelfEfficacy', 'Perceived_Ease_of_Use'),

    ('COVID_Impact', 'Perceived_Usefulness'),
    ('Compatibility', 'Perceived_Usefulness'),
    ('Information_Quality', 'Perceived_Usefulness'),
    ('System_Quality', 'Perceived_Usefulness'),
	('Student_Comp_SelfEfficacy', 'Perceived_Usefulness'),
    ('Perceived_Ease_of_Use', 'Perceived_Usefulness'),

    ('Perceived_Usefulness', 'Behavioral_Intention'),
    ('Perceived_Ease_of_Use', 'Behavioral_Intention'),

    ('Behavioral_Intention', 'Actual_Use')
])

# Defining the CPDs
# COVID Impact
cpd_covid = TabularCPD(
    variable='COVID_Impact', variable_card=3,
    values=cov_list
)

# Compatibility
cpd_comp = TabularCPD(
    variable='Compatibility', variable_card=3,
    values=com_list
)

# Information Quality
cpd_info_quality = TabularCPD(
    variable='Information_Quality', variable_card=3,
    values=iq_list
)

# System Quality
cpd_system_quality = TabularCPD(
    variable='System_Quality', variable_card=3,
    values=sq_list
)

# Student Computer Self-Efficacy
cpd_student_selfeff = TabularCPD(
    variable='Student_Comp_SelfEfficacy', variable_card=3,
    values=scse_list
)

# Perceived Ease of Use
peou_cpd = TabularCPD(
    variable='Perceived_Ease_of_Use',
    variable_card=3,
    values=[peou_low_values, peou_med_values, peou_high_values],
    evidence=['COVID_Impact', 'Compatibility', 'Information_Quality', 'System_Quality', 
	'Student_Comp_SelfEfficacy'],
    evidence_card=[3, 3, 3, 3, 3]
)

# Perceived Usefulness
pu_cpd = TabularCPD(
    variable='Perceived_Usefulness',
    variable_card=3,
    values=[PU_low_values, PU_med_values, PU_high_values],
    evidence=['COVID_Impact', 'Compatibility', 'Information_Quality', 'System_Quality', 
	'Student_Comp_SelfEfficacy', 'Perceived_Ease_of_Use'],
    evidence_card=[3, 3, 3, 3, 3, 3]
)

# Behavioral Intention
cpd_bi = TabularCPD(
    variable='Behavioral_Intention',
    variable_card=3,
    values=bi_lists,
    evidence=['Perceived_Usefulness', 'Perceived_Ease_of_Use'],
    evidence_card=[3, 3]
)

# Actual Use
cpd_au = TabularCPD(
    variable='Actual_Use',
    variable_card=3,
    values=au_lists,
    evidence=['Behavioral_Intention'],
    evidence_card=[3]
)

# Add CPDs to the model
model.add_cpds(cpd_covid, cpd_comp, cpd_info_quality, cpd_system_quality, cpd_student_selfeff, peou_cpd, pu_cpd, cpd_bi, cpd_au)
model.check_model()
inference = VariableElimination(model)

	
vars_list1 = [
    'COVID_Impact', 'Compatibility',
    'Information_Quality', 'System_Quality',
    'Student_Comp_SelfEfficacy',
    'Perceived_Usefulness', 'Perceived_Ease_of_Use',
    'Behavioral_Intention', 'Actual_Use'
]

# Marginal Probability of all the variables
for var in vars_list1:
    posterior = inference.query(variables=[var])
    print(f"Marginal Probability of {var}:\n{posterior}\n")

print("----------------------------------------------------------")

vars_list2 = [
    'COVID_Impact', 'Compatibility',
    'Information_Quality', 'System_Quality',
    'Student_Comp_SelfEfficacy',
    'Perceived_Usefulness', 'Perceived_Ease_of_Use',
    'Behavioral_Intention'
]

# Inference of setting Actual Use to High
for var in vars_list2:
    posterior = inference.query(variables=[var], evidence={'Actual_Use': 2})
    print(f"Posterior of {var} given Actual Use = High:\n{posterior}\n")

Marginal Probability of COVID_Impact:
+-----------------+---------------------+
| COVID_Impact    |   phi(COVID_Impact) |
+=================+=====================+
| COVID_Impact(0) |              0.0443 |
+-----------------+---------------------+
| COVID_Impact(1) |              0.7184 |
+-----------------+---------------------+
| COVID_Impact(2) |              0.2373 |
+-----------------+---------------------+

Marginal Probability of Compatibility:
+------------------+----------------------+
| Compatibility    |   phi(Compatibility) |
+==================+======================+
| Compatibility(0) |               0.0897 |
+------------------+----------------------+
| Compatibility(1) |               0.7415 |
+------------------+----------------------+
| Compatibility(2) |               0.1688 |
+------------------+----------------------+

Marginal Probability of Information_Quality:
+------------------------+----------------------------+
| Information_Quality    |   phi(Information_Q

# Subset Analysis


In [12]:
# This code checks how different combinations of parent variables affects our target node (Actual Use)
# parent variables
parents = [
    'COVID_Impact',
    'Compatibility',
    'Information_Quality',
    'System_Quality',
    'Student_Comp_SelfEfficacy'
]

results = []

subset_list = []

# Generate all subsets of size 2–5
for r in [2, 3, 4, 5]:
    for s in itertools.combinations(parents, r):
        subset_list.append(list(s))

# go through all subsets
for subset in subset_list:

    # Generate all state combinations for this subset (0=Low,1=Medium,2=High)
    combinations = list(itertools.product([0, 1, 2], repeat=len(subset)))

    for comb in combinations:

        state_no = {} # States: 0, 1, 2
        state_text = {} # States in text format: low, medium, high

        # convert numeric states to human-readable states (low, medium, high)
        for i in range(len(subset)):
            var_name = subset[i]
            var_value = comb[i]
            state_no[var_name] = var_value
            state_text[var_name] = labels[var_value]

        # compute P(Actual_Use = high | evidence)
        q = inference.query(['Actual_Use'], evidence=state_no)
        prob_high = q.values[2]

        # Save combination results
        row = {}
        for k in state_text:
            row[k] = state_text[k]

        row["P(Actual_Use=High)"] = prob_high
        results.append(row)

df = pd.DataFrame(results)
df_sorted = df.sort_values(by="P(Actual_Use=High)", ascending=False)
df_sorted.to_csv("results.csv", index=False)